In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration
from sklearn.model_selection import train_test_split
import re
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

In [ ]:
train_data = pd.read_csv('/content/drive/MyDrive/AI-ML-DL/samsum-train.csv')
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [ ]:
train_data.shape

(14732, 3)

In [ ]:
val_data = pd.read_csv('/content/drive/MyDrive/AI-ML-DL/samsum-validation.csv')
val_data.head()

,id,dialogue,summary
0,13817023,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...
1,13716628,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,13829420,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...
3,13819648,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.
4,13728448,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...


In [ ]:
val_data.shape

(818, 3)

In [ ]:
# random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [ ]:
train_data.shape

(4000, 3)

Data Preprocessing

In [ ]:
def clean_data(text):
  text = re.sub(r"\r\n"," ",text) # lines
  text = re.sub(r"\s+"," ",text) # spaces
  text = re.sub(r"<.*?>"," ",text) # html tags
  text = text.strip().lower()
  return text

In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

In [ ]:
val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

Tokenization

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")
# T5Tokenizer -- tokenizer class specifically designed for the T5 (Text-to-Text Transfer Transformer) model.
# from_pretrained -- loads a pre-trained tokenizer from the Hugging Face library.
# t5-small -- pretrained T5 model

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
def tokenize(data):
  inputs = tokenizer("summarize: "+data['dialogue'],padding="max_length",max_length=512, truncation=True)
  targets = tokenizer(data['summary'],padding="max_length",max_length=80, truncation=True)

  inputs["labels"] = targets["input_ids"] # token ids => add to inputs as labels
  return inputs

In [ ]:
train_dataset = train_data.apply(tokenize,axis=1).tolist() # axis =1 --row wise
val_dataset = val_data.apply(tokenize,axis=1).tolist()

In [ ]:
train_dataset[0]

{'input_ids': [21603, 10, 25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

# Train_dataset contains:

## input ids -> dialogue -> token ids
## 1 -> end of sequence, 0 -> padding
## attention mask -- shows valid & padding values i.e 1 - valid, 0-padding
## labels -> target -> summary token



In [ ]:
len(train_dataset[0]['input_ids'])

512

In [ ]:
type(train_dataset)

list

# Working with our Model

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
# setting up device

import torch

if torch.backends.mps.is_available():
  device = torch.device("mps")
  print("Using MPS")
elif torch.cuda.is_available():
  device = torch.device("cuda")
  print("Using CUDA")
else:
  device = torch.device("cpu")

print(device)
model.to(device)

Using CUDA
cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
# defining training arguments
train_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/AI-ML-DL/results",
    learning_rate=3e-5,
    num_train_epochs=8,
    predict_with_generate=True,
    weight_decay=0.01, # learns general patterns from the data
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch", #Evaluate the model after every epoch.
    save_strategy="epoch", # save model after every epoch
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    fp16=torch.cuda.is_available()
)


In [ ]:
# Seq2SeqTrainer -- class that automatically handles the entire seq training process of your model.
trainer = Seq2SeqTrainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer
)

# Model Training

In [22]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.770016,0.697458
2,0.725303,0.674675
3,0.708743,0.663695
4,0.702679,0.656854
5,0.674040,0.654136
6,0.698781,0.651205
7,0.640542,0.650458


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,0.770016,0.697458
2,0.725303,0.674675
3,0.708743,0.663695
4,0.702679,0.656854
5,0.674040,0.654136
6,0.698781,0.651205
7,0.640542,0.650458
8,0.649306,0.650536


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=4000, training_loss=0.7647264633178711, metrics={'train_runtime': 993.5381, 'train_samples_per_second': 32.208, 'train_steps_per_second': 4.026, 'total_flos': 4330937647104000.0, 'train_loss': 0.7647264633178711, 'epoch': 8.0})

# Save Model & Load Model

In [23]:
# save the model
model.save_pretrained("/content/drive/MyDrive/AI-ML-DL/results/saved_summary_model")
tokenizer.save_pretrained("/content/drive/MyDrive/AI-ML-DL/results/saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/AI-ML-DL/results/saved_summary_model/tokenizer_config.json',
 '/content/drive/MyDrive/AI-ML-DL/results/saved_summary_model/tokenizer.json')

In [24]:
# load the model
model= T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/AI-ML-DL/results/saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("/content/drive/MyDrive/AI-ML-DL/results/saved_summary_model")
print("Tokenizer loaded successfully!")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Tokenizer loaded successfully!


#Test the Model

In [25]:
def summarize_dialogue(dialogue):
  dialogue = "summarize: " + clean_data(dialogue) # clean

  # tokenize dialogue
  inputs = tokenizer(dialogue,padding="max_length",max_length=512,truncation=True,
                     return_tensors="pt").to(device)

  # generate the summary => token ids
  model.to(device)
  targets = model.generate(input_ids = inputs["input_ids"],
                           attention_mask = inputs["attention_mask"],
                           max_length=80,
                           min_length=15,
                           num_beams = 6, # Think of 6 possible summaries and choose the best one.
                           length_penalty=2.5,
                           no_repeat_ngram_size=3,
                           early_stopping=True)

  # convert targets token ids into summary -> decoding
  summary = tokenizer.decode(targets[0],skip_special_tokens=True)
  return summary

In [26]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)
print("Summary: ",summary)

Summary:  reports suggest that ai adoption has significantly increased over the past few years. experts are looking ahead to ensure that the systems are developed and used in a safe and beneficial way.
